# init

> Router initialization for the structure decomposition workflow

In [ ]:
#| default_exp routes.init

In [ ]:
#| export
from typing import List

from cjm_fasthtml_app_core.core.routing import APIRouter

# Import subpackage router assemblies
from cjm_fasthtml_workflow_transcript_decomp.routes.core.init import init_core_routers
from cjm_transcript_source_select.routes.init import init_selection_routers
from cjm_transcript_review.routes.init import init_review_routers
from cjm_transcript_verify.routes.init import init_verify_routers

# Consolidated segment-align init
from cjm_transcript_segment_align.routes.init import init_segment_align_routers

from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

## Router Assembly

The `init_routers` function calls all focused router initializers and wires up cross-router dependencies.

In [ ]:
#| export
def init_routers(
    workflow: "StructureDecompWorkflow",  # The workflow instance
) -> List[APIRouter]:  # List of configured routers
    """Initialize and return all workflow routers."""
    base_prefix = workflow.config.route_prefix

    # Core routers (status, sources, audio)
    core_routers, core_routes = init_core_routers(
        workflow, f"{base_prefix}/core"
    )
    
    # Selection routers
    selection = init_selection_routers(
        state_store=workflow.state_store,
        source_service=workflow.source_service,
        workflow_id=workflow.config.workflow_id,
        prefix=f"{base_prefix}/selection",
    )

    # Segment & Align routers (consolidated)
    sa_routers, sa_result = init_segment_align_routers(
        state_store=workflow.state_store,
        workflow_id=workflow.config.workflow_id,
        prefix=base_prefix,
        source_service=workflow.source_service,
        plugin_manager=workflow.plugin_manager,
        job_queue=workflow.job_queue,
        audio_src_url=core_routes["audio_src"].to(),
        text_plugin=workflow.config.text_plugin,
        vad_plugin=workflow.config.vad_plugin,
        fa_plugin_name=workflow.config.fa_plugin_name,
        sysmon_plugin_name=workflow.config.sysmon_plugin_name,
        max_history_depth=workflow.config.max_history_depth,
    )

    # Review routers
    review_routers, review_urls, review_routes = init_review_routers(
        state_store=workflow.state_store,
        workflow_id=workflow.config.workflow_id,
        prefix=f"{base_prefix}/review",
        audio_src_url=core_routes["audio_src"].to(),
        graph_service=workflow.graph_service,
        alert_container_id="commit-alert-container",
    )
    
    # Verify routers
    verify_routers, verify_urls, verify_routes = init_verify_routers(
        state_store=workflow.state_store,
        workflow_id=workflow.config.workflow_id,
        prefix=f"{base_prefix}/verify",
        verify_service=workflow.verify_service,
    )

    # Store on workflow for renderer access
    workflow._selection_result = selection
    workflow._selection_urls = selection.urls
    workflow._render_local_files_panel = selection.render_local_files_panel
    workflow._sb_state = selection.sb_state
    workflow._fb_kb_manager = selection.fb_kb_manager
    workflow._sa_result = sa_result
    workflow._review_urls = review_urls
    workflow._verify_urls = verify_urls
    workflow._core_routes = core_routes

    return (
        core_routers + sa_routers +
        selection.routers + review_routers + verify_routers
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()